In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import helpers
from compression import zstd

In [ ]:
x_train, x_test, y_train, train_ids, test_ids = helpers.load_data("../data")

In [ ]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)

## Exploratory Data Analysis

In [ ]:

nan_frac = np.isnan(x_train).mean(axis=0)
nan_sort = np.sort(nan_frac, axis=0)

plt.bar(np.arange(x_train.shape[1]), nan_sort*100)
plt.xlabel("Column Index")
plt.ylabel("%NaN")
plt.show()

A lot of columns are majority NaN. We should pich a threshold to drop these columns.<br/>

In [ ]:
with zstd.open("../data/x_train.csv.zst", "rt") as f:
    header = f.readline()
col_names = np.array(header.strip().split(",")[1:])
idx = {name: j for j, name in enumerate(col_names)}

In [ ]:
n_unique = np.array([len(np.unique(c[~np.isnan(c)])) for c in x_train.T])

def summary(j):
    c = x_train[:, j]; c = c[~np.isnan(c)]
    vals, counts = np.unique(c, return_counts=True)
    top = vals[np.argsort(counts)[::-1][:5]]
    return f"{col_names[j]:<10} nan={nan_frac[j]:.0%} uniq={n_unique[j]:<5} min={c.min():g} max={c.max():g} top={top}"

for j in np.argsort(nan_frac):
    print(summary(j))


Some columns have weird values such as [88. 14.  2. 77.  3.] for ADDOWN, ADSLEEP etc. Looking at the documentation of the [dataset](https://www.cdc.gov/brfss/annual_data/annual_2022.html), we can see that certain columns use codes such as:
- 7 / 77 / 777 / 7777: don't know / not sure
- 9 / 99 / 999 / 9999: refused

## Relationship with positive cases 

We find the rate of positive and negative cases:

In [ ]:
pos = y_train == 1
print(f"base rate: {pos.mean():.3f}")

For each categorical column (2 to 10 distinct values), we split the rows by their answer and compute the positive rate in each group. For example, for CVDSTRK3 (ever had a stroke?), people who answered YES are positive far more often than 8.8%, while those who answered NO are below it. A column like the interview month gives ~8.8% in every group, so it carries no signal.

We measure this with [Cramér's V](https://en.wikipedia.org/wiki/Cram%C3%A9r%27s_V), the effect size of [Pearson's chi-squared test of independence](https://en.wikipedia.org/wiki/Pearson%27s_chi-squared_test) between the column and the label. With a binary label, the chi-squared statistic simplifies to
$$\chi^2 = \sum_v \frac{n_v\,(r_v - p)^2}{p(1-p)}, \qquad V = \sqrt{\chi^2 / N}$$

In [ ]:
pos = y_train == 1
print(f"base rate: {pos.mean():.3f}")

rows = []
for j in np.where((n_unique > 1) & (n_unique <= 10) & (nan_frac < 0.8))[0]:
    c = x_train[:, j]
    ok = ~np.isnan(c)
    p = pos[ok].mean()
    if not 0 < p < 1:
        continue
    vals, inv, counts = np.unique(c[ok], return_inverse=True, return_counts=True)
    rates = np.bincount(inv, weights=pos[ok]) / counts
    chi2 = np.sum(counts * (rates - p) ** 2) / (p * (1 - p))
    v = np.sqrt(chi2 / ok.sum())  # Cramer's V
    rows.append((v, j, vals, rates, counts))

for v, j, vals, rates, counts in sorted(rows, key=lambda r: -r[0])[:20]:
    desc = " ".join(f"{val:g}:{r:.2f}({n // 1000}k)" for val, r, n in zip(vals, rates, counts))
    print(f"{col_names[j]:<10} V={v:.3f} nan={nan_frac[j]:.0%}  {desc}")


For continuous columns: standardized mean difference between classes, d = (mean(y=1) - mean(y=-1)) / std.

In [ ]:
rows = []
for j in np.where((n_unique > 10) & (nan_frac < 0.8))[0]:
    c = x_train[:, j]
    ok = ~np.isnan(c)
    d = (c[ok & pos].mean() - c[ok & ~pos].mean()) / c[ok].std()
    rows.append((abs(d), j, d))
    

for _, j, d in sorted(rows, reverse=True)[:15]:
    print(f"{col_names[j]} d={d}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, name in zip(axes.flat, ["_AGE80", "PHYSHLTH", "_BMI5", "GENHLTH"]):
    c = x_train[:, idx[name]]
    ok = ~np.isnan(c)
    bins = np.histogram_bin_edges(c[ok], bins=30)
    ax.hist(c[ok & ~pos], bins=bins, density=True, alpha=0.5, label="y = -1")
    ax.hist(c[ok & pos], bins=bins, density=True, alpha=0.5, label="y = 1")
    ax.set_title(name)
    ax.set_ylabel("density")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

## Conditioning on NaN

Some questions questions are only asked if a previous answer triggers them (e.g. the diabetes module is only asked to diabetics). Therefore we check whether:
$$P(Y = 1 \mid \text{NaN})$$
 
is informative for each column

In [ ]:
rows = []
for j in range(x_train.shape[1]):
    m = np.isnan(x_train[:, j])
    if 0 < m.sum():
        rows.append((abs(pos[m].mean() - pos[~m].mean()), j, pos[m].mean(), pos[~m].mean(), (~m).sum()))

for _, j, p_nan, p_present, n in sorted(rows, reverse=True)[:15]:
    print(f"{col_names[j]} present={n} P(y=1|nan)={p_nan}  P(y=1|present)={p_present}")

HAREHAB1 is the code for had heart attack before, this could be a leak?

## Feature Engineering:

1. Number of diseases

In [ ]:
# Health conditions we count, all coded 1 = "yes"
CONDITIONS = {
    "CVDSTRK3": "stroke",
    "BPHIGH4": "high blood pressure",
    "TOLDHI2": "high cholesterol",
    "DIABETE3": "diabetes",
    "CHCKIDNY": "kidney disease",
    "CHCCOPD1": "COPD",
    "HAVARTH3": "arthritis",
    "DIFFWALK": "difficulty walking",
}

# For each respondent, find the num of diseases
n_conditions = np.zeros(len(x_train))
for column in CONDITIONS:
    n_conditions += x_train[:, idx[column]] == 1

# Share of people with MICHD for each number of conditions
levels = np.unique(n_conditions)
pct_positive = []
for k in levels:
    group = n_conditions == k
    pct_positive.append(pos[group].mean() * 100)


plt.bar(levels, pct_positive)
plt.xlabel("number of conditions reported")
plt.ylabel("% with MICHD")
plt.title("Risk grows with the number of conditions")
plt.show()


2. Interaction: sex × agem

In [ ]:
sex = x_train[:, idx["SEX"]]
age = x_train[:, idx["_AGEG5YR"]] 
ages = np.arange(1, 14)
for s, label in [(1, "male"), (2, "female")]:
    plt.plot(ages, [pos[(sex == s) & (age == a)].mean() * 100 for a in ages], marker="o", label=label)
plt.xticks(ages)
plt.xlabel("age group")
plt.ylabel("% with MICHD")
plt.legend()
plt.show()